In [4]:
import pandas as pd

file_path = r'C:\Users\User\OneDrive - Universiti Malaya\UM SEM 2\WQF7007 - NATURAL LANGUAGE PROCESSING\fake_job_postings.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}. First 5 rows:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the path is correct and the file exists in your Google Drive.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

Successfully loaded data from C:\Users\User\OneDrive - Universiti Malaya\UM SEM 2\WQF7007 - NATURAL LANGUAGE PROCESSING\fake_job_postings.csv. First 5 rows:


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [5]:
df.shape

(17880, 18)

In [6]:
df.isnull().sum()

job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64

In [7]:
drop_cols = ['salary_range', 'department']
df = df.drop(columns=drop_cols)

In [8]:
df.shape

(17880, 16)

In [9]:
text_cols_main = [
    'title', 'description', 'requirements',
    'company_profile', 'benefits'
]

text_cols_secondary = [
    'location', 'industry', 'function'
]

categorical_cols = [
    'employment_type', 'required_experience', 'required_education'
]

binary_cols = [
    'telecommuting', 'has_company_logo', 'has_questions'
]

#Excluded job_id & fradulent
# job_id -> Identidifer
# fradulent -> Label

In [10]:
for col in text_cols_main:
    df[col] = df[col].fillna("missing")

for col in text_cols_secondary:
    df[col] = df[col].fillna("unknown")

for col in categorical_cols:
    df[col] = df[col].fillna("not_specified")

for col in binary_cols:
    df[col] = df[col].astype(int)

# Secondary text → "unknown"
# Categorical → "not_specified"
# Binary → ensure integer

In [11]:
#Feature Engineering
# Missing in 'requirements'/'company_profile'/'benefits' -> suspicious?
df['missing_requirements'] = (df['requirements'] == "missing").astype(int)
df['missing_company_profile'] = (df['company_profile'] == "missing").astype(int)
df['missing_benefits'] = (df['benefits'] == "missing").astype(int)

# Has numbers feature
# Has numbers in 'descriptio' -> suspicious
df['has_number'] = df['description'].str.contains(r'\d').astype(int)

# Text length feature
# Fake job -> short/vague
# Real job -> detailed 'description'
df['desc_length'] = df['description'].apply(len)

In [12]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)          # remove newline
    text = re.sub(r'\s+', ' ', text)         # remove extra spaces
    return text.strip()

for col in text_cols_main + text_cols_secondary:
    df[col] = df[col].apply(clean_text)

In [13]:
# COMBINE TEXT FOR BERT / RoBERTa

df['full_text'] = (
    "TITLE: " + df['title'] + " " +
    "DESC: " + df['description'] + " " +
    "REQ: " + df['requirements'] + " " +
    "COMPANY: " + df['company_profile'] + " " +
    "BENEFITS: " + df['benefits'] + " " +
    "INDUSTRY: " + df['industry'] + " " +
    "FUNCTION: " + df['function']
)

In [14]:
print("\nMissing values after preprocessing:")
print(df.isnull().sum())


Missing values after preprocessing:
job_id                     0
title                      0
location                   0
company_profile            0
description                0
requirements               0
benefits                   0
telecommuting              0
has_company_logo           0
has_questions              0
employment_type            0
required_experience        0
required_education         0
industry                   0
function                   0
fraudulent                 0
missing_requirements       0
missing_company_profile    0
missing_benefits           0
has_number                 0
desc_length                0
full_text                  0
dtype: int64


In [15]:
print("\nSample processed text:")
print(df['full_text'].iloc[0][:500])


Sample processed text:
TITLE: marketing intern DESC: food52, a fast-growing, james beard award-winning online food community and crowd-sourced and curated recipe hub, is currently interviewing full- and part-time unpaid interns to work in a small team of editors, executives, and developers in its new york city headquarters.reproducing and/or repackaging existing food52 content for a number of partner sites, such as huffington post, yahoo, buzzfeed, and more in their various content management systemsresearching blogs 
